# 03 — MILP Robustness Verification

Formally verifies the L∞ robustness of a trained ReLU MLP using Mixed-Integer
Linear Programming (MILP).

**Algorithm:**
1. Compute IBP (Interval Bound Propagation) pre-activation bounds for the MLP.
2. Encode each ReLU with a big-M binary variable using the IBP-tight bounds.
3. For each class `c ≠ y`, maximise `logit_c - logit_y` over the L∞ ball.
4. If the worst margin ≤ 0 across all classes → **VERIFIED** (no adversarial example exists).
   Otherwise → **FALSIFIED** (extract the adversarial point).

**Thesis context — WP2:** Exact (complete) robustness certificates for the baseline MLP.

**Solver:** Uses Gurobi if `gurobipy` is available, otherwise falls back to OR-Tools CBC.

**Prerequisites:** Run `01_train_mnist.ipynb` first (or point `CKPT_PATH` to an existing checkpoint).

In [ ]:
!pip install -q torch torchvision numpy pandas ortools

## 1 — Library code: types, models, data utilities

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn, Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# ── Verification result types ──────────────────────────────────────────────────
class VerificationStatus(str, Enum):
    VERIFIED  = "VERIFIED"
    FALSIFIED = "FALSIFIED"
    TIMEOUT   = "TIMEOUT"
    ERROR     = "ERROR"


@dataclass
class VerificationResult:
    status:       VerificationStatus
    worst_margin: float | None
    solve_time:   float | None
    solver_name:  str
    adv_example:  list[float] | None


# ── LinExpr (symbolic linear expression for MILP) ─────────────────────────────
@dataclass
class LinExpr:
    terms: Dict[Any, float]
    const: float = 0.0


def expr_const(c: float) -> LinExpr:  return LinExpr(terms={}, const=float(c))
def expr_var(v, coeff=1.0) -> LinExpr: return LinExpr(terms={v: float(coeff)}, const=0.0)

def expr_add(a: LinExpr, b: LinExpr) -> LinExpr:
    terms = dict(a.terms)
    for v, c in b.terms.items():
        terms[v] = terms.get(v, 0.0) + c
    return LinExpr(terms=terms, const=a.const + b.const)

def expr_mul(a: LinExpr, scalar: float) -> LinExpr:
    s = float(scalar)
    return LinExpr(terms={v: c * s for v, c in a.terms.items()}, const=a.const * s)

def expr_sub(a: LinExpr, b: LinExpr) -> LinExpr:
    return expr_add(a, expr_mul(b, -1.0))


# ── Data ───────────────────────────────────────────────────────────────────────
def get_mnist_datasets(data_dir="data"):
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(root=str(data_dir), train=True,  download=True, transform=tfm)
    test_ds  = datasets.MNIST(root=str(data_dir), train=False, download=True, transform=tfm)
    return train_ds, test_ds


@dataclass(frozen=True)
class Split:
    seed: int
    indices: list[int]


def load_split(path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])


def ensure_mnist_eval_split(path, seed=1234, n=100) -> Split:
    p = Path(path)
    if p.exists(): return load_split(p)
    indices = random.Random(seed).sample(range(10_000), n)
    split = Split(seed=seed, indices=indices)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({"seed": split.seed, "indices": split.indices}, indent=2) + "\n", encoding="utf-8")
    return split


# ── Models ─────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(int(in_dim), int(h1))
        self.fc2 = nn.Linear(int(h1), int(h2))
        self.fc3 = nn.Linear(int(h2), int(num_classes))
        self.relu = nn.ReLU()

    def forward(self, x):
        if x.ndim == 4: x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

    def linear_layers(self): return [self.fc1, self.fc2, self.fc3]


# ── Checkpoint I/O ─────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw  = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    model = MnistMlp(**meta.model_kwargs) if meta.model_type == "mlp" else None
    if model is None: raise ValueError(f"Unsupported model type: {meta.model_type!r}")
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


print("Core library code loaded")

## 2 — MILP components: IBP bounds, encoding, solver backends

In [ ]:
# ── IBP bounds ─────────────────────────────────────────────────────────────────
def compute_mlp_ibp_bounds(model: nn.Module, x_center: Tensor, eps: float) -> dict:
    """Compute interval bound propagation pre-activation bounds for an MLP."""
    x0  = x_center.detach().view(-1)
    x_l = torch.clamp(x0 - float(eps), 0.0, 1.0)
    x_u = torch.clamp(x0 + float(eps), 0.0, 1.0)

    h_l_list = [x_l]
    h_u_list = [x_u]
    a_l_list, a_u_list = [], []

    linears = list(model.linear_layers())
    for li, layer in enumerate(linears):
        W = layer.weight.detach()
        b = layer.bias.detach()
        W_plus  = torch.clamp(W, min=0.0)
        W_minus = torch.clamp(W, max=0.0)
        a_l = W_plus @ h_l_list[-1] + W_minus @ h_u_list[-1] + b
        a_u = W_plus @ h_u_list[-1] + W_minus @ h_l_list[-1] + b
        a_l_list.append(a_l)
        a_u_list.append(a_u)
        if li < len(linears) - 1:
            h_l_list.append(torch.relu(a_l))
            h_u_list.append(torch.relu(a_u))

    return {"x_l": x_l, "x_u": x_u, "a_l_list": a_l_list, "a_u_list": a_u_list}


# ── OR-Tools CBC backend ───────────────────────────────────────────────────────
class OrtoolsCbcBackend:
    solver_name = "CBC"

    def __init__(self):
        from ortools.linear_solver import pywraplp
        self._pywraplp = pywraplp
        solver = pywraplp.Solver.CreateSolver("CBC")
        if solver is None:
            solver = pywraplp.Solver.CreateSolver("CBC_MIXED_INTEGER_PROGRAMMING")
        if solver is None:
            raise RuntimeError("Failed to create CBC solver via OR-Tools.")
        self.solver = solver
        self._obj_expr = None

    def add_var(self, lb, ub, binary=False):
        return self.solver.IntVar(0.0, 1.0, "") if binary else self.solver.NumVar(float(lb), float(ub), "")

    def add_binary(self):
        return self.add_var(0.0, 1.0, binary=True)

    def _to_expr(self, e: LinExpr):
        lin = self.solver.Sum([coeff * var for var, coeff in e.terms.items()])
        if e.const: lin = lin + float(e.const)
        return lin

    def add_le(self, lhs, rhs): self.solver.Add(self._to_expr(expr_sub(lhs, rhs)) <= 0.0)
    def add_ge(self, lhs, rhs): self.solver.Add(self._to_expr(expr_sub(lhs, rhs)) >= 0.0)
    def add_eq(self, lhs, rhs): self.solver.Add(self._to_expr(expr_sub(lhs, rhs)) == 0.0)

    def set_objective_max(self, expr):
        self._obj_expr = expr
        self.solver.Maximize(self._to_expr(expr))

    def solve(self, time_limit_s=None):
        if time_limit_s is not None:
            self.solver.set_time_limit(int(max(time_limit_s, 0.0) * 1000))
        t0 = time.time()
        status = self.solver.Solve()
        elapsed = time.time() - t0
        pywraplp = self._pywraplp
        status_map = {
            pywraplp.Solver.OPTIMAL:    "OPTIMAL",
            pywraplp.Solver.FEASIBLE:   "FEASIBLE",
            pywraplp.Solver.INFEASIBLE: "INFEASIBLE",
            pywraplp.Solver.UNBOUNDED:  "UNBOUNDED",
        }
        status_str = status_map.get(status, "UNKNOWN")
        obj_val = None
        if self._obj_expr is not None and status in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            obj_val = float(self.solver.Objective().Value())
        return status_str, obj_val, elapsed

    def get_value(self, var) -> float: return float(var.solution_value())


# ── Gurobi backend (optional) ──────────────────────────────────────────────────
class GurobiBackend:
    solver_name = "Gurobi"

    def __init__(self):
        import gurobipy as gp
        self.gp = gp
        self.model = gp.Model()
        self.model.Params.OutputFlag = 0

    def add_var(self, lb, ub, binary=False):
        vtype = self.gp.GRB.BINARY if binary else self.gp.GRB.CONTINUOUS
        return self.model.addVar(lb=float(lb), ub=float(ub), vtype=vtype)

    def add_binary(self): return self.add_var(0, 1, binary=True)

    def _to_expr(self, e: LinExpr):
        lin = self.gp.LinExpr()
        for var, coeff in e.terms.items(): lin.add(var, float(coeff))
        if e.const: lin += float(e.const)
        return lin

    def add_le(self, lhs, rhs): self.model.addLConstr(self._to_expr(expr_sub(lhs, rhs)) <= 0.0)
    def add_ge(self, lhs, rhs): self.model.addLConstr(self._to_expr(expr_sub(lhs, rhs)) >= 0.0)
    def add_eq(self, lhs, rhs): self.model.addLConstr(self._to_expr(expr_sub(lhs, rhs)) == 0.0)

    def set_objective_max(self, expr):
        self.model.setObjective(self._to_expr(expr), sense=self.gp.GRB.MAXIMIZE)

    def solve(self, time_limit_s=None):
        if time_limit_s is not None: self.model.Params.TimeLimit = float(time_limit_s)
        t0 = time.time()
        self.model.optimize()
        elapsed = time.time() - t0
        status_map = {
            self.gp.GRB.OPTIMAL:    "OPTIMAL",
            self.gp.GRB.SUBOPTIMAL: "FEASIBLE",
            self.gp.GRB.TIME_LIMIT: "TIME_LIMIT",
            self.gp.GRB.INFEASIBLE: "INFEASIBLE",
            self.gp.GRB.UNBOUNDED:  "UNBOUNDED",
        }
        status_str = status_map.get(self.model.Status, "UNKNOWN")
        obj_val = float(self.model.objVal) if self.model.SolCount > 0 else None
        return status_str, obj_val, elapsed

    def get_value(self, var) -> float: return float(var.X)


def get_backend(name: str):
    lname = name.lower()
    if lname == "cbc":    return OrtoolsCbcBackend()
    if lname == "gurobi": return GurobiBackend()
    if lname == "auto":
        try:   return GurobiBackend()
        except Exception: return OrtoolsCbcBackend()
    raise ValueError(f"Unknown backend {name!r}")


print("MILP backends loaded")

## 3 — MILP encoder and verifier

In [ ]:
def _affine_layer(backend, W, b, prev_h_vars):
    """Add equality constraints for one affine layer: a = W*h + b."""
    out_dim, in_dim = W.shape
    a_vars = [backend.add_var(lb=-1e9, ub=1e9) for _ in range(out_dim)]
    for i in range(out_dim):
        rhs = expr_const(float(b[i].item()))
        for j in range(in_dim):
            coef = float(W[i, j].item())
            if coef != 0.0:
                rhs = expr_add(rhs, expr_mul(expr_var(prev_h_vars[j]), coef))
        backend.add_eq(expr_var(a_vars[i]), rhs)
    return a_vars


def _relu_layer(backend, a_vars, a_l, a_u):
    """Add big-M ReLU constraints: h = max(0, a)."""
    h_vars = []
    for i, a_var in enumerate(a_vars):
        l, u = float(a_l[i].item()), float(a_u[i].item())
        if u <= 0.0:
            h = backend.add_var(lb=0.0, ub=0.0)
        elif l >= 0.0:
            h = backend.add_var(lb=l, ub=u)
            backend.add_eq(expr_var(h), expr_var(a_var))
        else:
            h     = backend.add_var(lb=0.0, ub=u)
            b_bin = backend.add_binary()
            backend.add_ge(expr_var(h), expr_const(0.0))
            backend.add_ge(expr_var(h), expr_var(a_var))
            one_minus_b = expr_add(expr_const(1.0), expr_mul(expr_var(b_bin), -1.0))
            backend.add_le(expr_var(h), expr_add(expr_var(a_var), expr_mul(one_minus_b, -l)))
            backend.add_le(expr_var(h), expr_mul(expr_var(b_bin), u))
            backend.add_ge(expr_var(a_var), expr_const(l))
            backend.add_le(expr_var(a_var), expr_const(u))
        h_vars.append(h)
    return h_vars


def encode_mlp_on_box(backend, model, x_center, eps):
    """Encode the MLP as MILP constraints over the L-inf ball around x_center."""
    bounds   = compute_mlp_ibp_bounds(model, x_center, eps)
    x_l, x_u = bounds["x_l"], bounds["x_u"]
    x0_flat  = x_center.detach().view(-1)

    in_vars = [backend.add_var(lb=float(x_l[i]), ub=float(x_u[i])) for i in range(x0_flat.numel())]
    linears = list(model.linear_layers())
    prev_h_vars = in_vars
    logit_vars  = []

    for li, layer in enumerate(linears):
        W = layer.weight.detach()
        b = layer.bias.detach()
        a_vars = _affine_layer(backend, W=W, b=b, prev_h_vars=prev_h_vars)
        if li == len(linears) - 1:
            logit_vars = a_vars
        else:
            prev_h_vars = _relu_layer(backend, a_vars, bounds["a_l_list"][li], bounds["a_u_list"][li])

    return {"input_vars": in_vars, "logit_vars": logit_vars}


def _status_from_str(s: str) -> VerificationStatus:
    if s in {"OPTIMAL", "FEASIBLE"}:    return VerificationStatus.FALSIFIED
    if s in {"INFEASIBLE", "UNBOUNDED"}: return VerificationStatus.VERIFIED
    if s == "TIME_LIMIT":               return VerificationStatus.TIMEOUT
    return VerificationStatus.ERROR


def verify_point(model, x, y: int, eps: float, backend_name="auto", time_limit_s=None):
    """Verify robustness of model at point x with label y under L-inf eps."""
    device = next(model.parameters()).device
    x = x.detach().to(device)

    worst_margin = float("-inf")
    best_adv     = None
    t_accum      = 0.0
    last_solver  = ""

    for c in [ci for ci in range(10) if ci != int(y)]:
        backend  = get_backend(backend_name)
        encoded  = encode_mlp_on_box(backend, model=model, x_center=x, eps=eps)
        margin   = expr_sub(expr_var(encoded["logit_vars"][c]), expr_var(encoded["logit_vars"][int(y)]))
        backend.set_objective_max(margin)

        status_str, obj_val, t_s = backend.solve(time_limit_s=time_limit_s)
        last_solver = backend.solver_name
        t_accum    += float(t_s)
        status      = _status_from_str(status_str)

        if status in {VerificationStatus.ERROR, VerificationStatus.TIMEOUT}:
            return VerificationResult(status=status, worst_margin=None,
                                      solve_time=t_accum, solver_name=last_solver, adv_example=None)

        if obj_val is not None and obj_val > worst_margin:
            worst_margin = float(obj_val)
            if status == VerificationStatus.FALSIFIED:
                best_adv = [backend.get_value(v) for v in encoded["input_vars"]]

    if worst_margin <= 0.0:
        return VerificationResult(status=VerificationStatus.VERIFIED, worst_margin=worst_margin,
                                  solve_time=t_accum, solver_name=last_solver, adv_example=None)
    return VerificationResult(status=VerificationStatus.FALSIFIED, worst_margin=worst_margin,
                              solve_time=t_accum, solver_name=last_solver, adv_example=best_adv)


print("MILP verifier defined")

## 4 — Configuration

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
CKPT_PATH   = "runs/mlp_mnist/model.pt"            # path to trained MLP checkpoint
SUBSET_PATH = "assets/splits/mnist_eval_100.json"  # fixed evaluation split
DATA_DIR    = "data"
EPS         = 0.03    # L-inf perturbation radius
SOLVER      = "auto"  # "auto", "cbc", or "gurobi"
TIME_LIMIT  = 30.0    # seconds per LP (per class)
MAX_SAMPLES = 5       # number of samples to verify (MILP is slow!)
DEVICE      = "cpu"   # MILP runs on CPU
# ──────────────────────────────────────────────────────────────────────────────

print(f"eps={EPS} | solver={SOLVER} | time_limit={TIME_LIMIT}s | max_samples={MAX_SAMPLES}")

## 5 — Load model & evaluation split

In [ ]:
model, meta, _ = load_checkpoint(CKPT_PATH, map_location=DEVICE)
model.to(DEVICE).eval()
print(f"Model: {meta.model_type!r}  run={meta.run_name!r}")

split   = ensure_mnist_eval_split(SUBSET_PATH)
indices = split.indices[:MAX_SAMPLES]
print(f"Verifying {len(indices)} samples from split (seed={split.seed})")

## 6 — Run MILP verification

This is the most compute-intensive step. Each sample requires solving 9 MILP
problems (one per class `c ≠ y`). Expect ~seconds–minutes per sample depending
on the model size and solver.

In [ ]:
_, test_ds = get_mnist_datasets(DATA_DIR)
sub_ds     = Subset(test_ds, indices)
loader     = DataLoader(sub_ds, batch_size=1, shuffle=False, num_workers=0)

rows    = []
adv_dir = Path("results/adversarial")
adv_dir.mkdir(parents=True, exist_ok=True)

for local_idx, (x, y) in enumerate(loader):
    x, y    = x.to(DEVICE), y.to(DEVICE)
    idx     = int(indices[local_idx])
    label   = int(y[0].item())

    print(f"[{local_idx+1}/{len(indices)}] sample idx={idx}  label={label} ...", end=" ", flush=True)
    res = verify_point(model=model, x=x[0], y=label, eps=EPS,
                       backend_name=SOLVER, time_limit_s=TIME_LIMIT)
    print(f"{res.status.value}  margin={res.worst_margin}  t={res.solve_time:.2f}s")

    adv_path_str = None
    if res.status == VerificationStatus.FALSIFIED and res.adv_example is not None:
        adv = np.array(res.adv_example, dtype=np.float32).reshape(1, 28, 28)
        adv_path = adv_dir / f"mnist_idx{idx}_eps{EPS:.4f}.npz"
        np.savez_compressed(adv_path, x_adv=adv, x0=x.cpu().numpy(), eps=float(EPS), y=int(label))
        adv_path_str = str(adv_path)

    rows.append({
        "index":        idx,
        "y":            label,
        "status":       res.status.value,
        "worst_margin": res.worst_margin,
        "time_s":       res.solve_time,
        "solver":       res.solver_name,
        "adv_path":     adv_path_str,
    })

df = pd.DataFrame(rows)
Path("results").mkdir(parents=True, exist_ok=True)
out_path = f"results/milp_{meta.run_name}_eps{EPS:.4f}.csv"
df.to_csv(out_path, index=False)
print(f"\nResults saved to: {out_path}")

## 7 — Summary

In [ ]:
print(df[["index", "y", "status", "worst_margin", "time_s", "solver"]].to_string(index=False))
print()
print("Status counts:")
print(df["status"].value_counts().to_string())
print(f"\nMean solve time: {df['time_s'].mean():.2f}s")